# UFO Witness Report Analyzer

Capstone project. Multi-class text classification.

Paste a witness report and the model predicts the most likely shape (triangle, disk, fireball, etc), plus confidence and a few keywords from the text.

| | |
|---|---|
| Task | Multi-class text classification |
| Features | TF-IDF on the report text |
| Model | Multinomial Naive Bayes |
| Data | ~80k public NUFORC sighting reports |

**Sammuel Hinton**  ·  ICT 890-015, Machine Learning & AI Bootcamp  ·  University of Calgary Continuing Education (delivered by RoboGarden)  
Instructors: Omar Ashinawy, Hesam Damghanian, Amber Carney  ·  with thanks to Mostafa Moharram

Live site: https://cdbz8a.github.io/ufo-witness-report-analyzer/  
Repo: https://github.com/cdbz8a/ufo-witness-report-analyzer

This is for class / educational use only. It does **not** prove any sighting is real.


## How to run in Colab

1. File → Upload notebook
2. Runtime → Run all
3. Wait for the data download + training (takes a couple minutes)
4. Scroll to the bottom and try your own report

Tip: if the download fails, re-run the load cell once. Colab does that sometimes.


## 1. Imports


In [ ]:
import re
import html

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    ConfusionMatrixDisplay,
)
from sklearn.metrics.pairwise import cosine_similarity

# makes the plots a bit nicer
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 100)

print("imports ok")


## 2. Load the data

This CSV has **no header row**, so I had to name the columns myself or pandas grabs the first report as the header.


In [ ]:
# public dataset
url = (
    "https://raw.githubusercontent.com/planetsig/ufo-reports/master/"
    "csv-data/ufo-scrubbed-geocoded-time-standardized.csv"
)

# no header in the csv so I name the columns
cols = [
    "datetime", "city", "state", "country", "shape",
    "duration_sec", "duration_text", "comments", "date_posted", "lat", "lon",
]

df = pd.read_csv(url, names=cols, low_memory=False)

# only keep the columns I need
df = df[["comments", "shape", "city", "state", "country", "datetime"]].copy()

# drop rows missing text or the label
df = df.dropna(subset=["comments", "shape"])

print("Total reports:", len(df))
print()
print(df["shape"].value_counts())


## 3. Clean the text and pick shapes

- fix HTML junk in the comments (like `&#44;` which is just a comma)
- lowercase the shape labels
- drop really short comments
- keep the 8 most common shapes
- drop `unknown` because its not really a shape


In [ ]:
def clean_text(text):
    # missing values
    if pd.isna(text):
        return ""
    # fix html junk like &#44; -> comma
    text = html.unescape(str(text))
    # fix weird spacing
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["comments"] = df["comments"].map(clean_text)
df["shape"] = df["shape"].astype(str).str.strip().str.lower()

# skip tiny comments, not useful
df = df[df["comments"].str.len() >= 10]

# keep shapes that show up enough times
keep_shapes = [
    "light", "triangle", "circle", "fireball",
    "sphere", "disk", "oval", "cigar",
]
df = df[df["shape"].isin(keep_shapes)].copy()
df = df.reset_index(drop=True)

# year for the similar reports thing later
df["year"] = pd.to_datetime(df["datetime"], errors="coerce").dt.year

print("Rows after filter:", len(df))
print()
print(df["shape"].value_counts())
df[["comments", "shape", "city", "state"]].head()


## 4. Quick look at the data


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# count per shape
order = df["shape"].value_counts().index
sns.countplot(data=df, y="shape", order=order, ax=axes[0], color="steelblue")
axes[0].set_title("Shape counts")
axes[0].set_xlabel("Count")

# comment lengths
df["comments"].str.len().hist(bins=30, ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_title("Comment length")
axes[1].set_xlabel("Characters")
axes[1].set_ylabel("Reports")

plt.tight_layout()
plt.show()

print(df["comments"].str.len().describe())


## 5. Train / test split + model

Pretty standard text classification pipeline:

1. split the text and labels (stratified so rare shapes stay in both sets)
2. TF-IDF turns words into numbers
3. Multinomial Naive Bayes does the classifying

8 classes total.


In [ ]:
X = df["comments"]  # text
y = df["shape"]     # labels / shapes

# keep location stuff for similar reports
X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X,
    y,
    df[["city", "state", "country", "year", "comments", "shape"]],
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# tf-idf then naive bayes
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),  # single words + bigrams
        min_df=3,            # ignore rare terms
        max_df=0.9,          # ignore super common terms
    )),
    ("clf", MultinomialNB(alpha=0.5)),
])

model.fit(X_train, y_train)
print("train:", len(X_train), " test:", len(X_test))


## 6. Evaluate

`light` is by far the biggest class, so accuracy alone can look better than the model really is. I also look at macro-F1 and the confusion matrix.


In [ ]:
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print("Accuracy   :", round(acc, 3))
print("F1 macro   :", round(f1_macro, 3))
print("F1 weighted:", round(f1_weighted, 3))
print()
print(classification_report(y_test, y_pred, zero_division=0))

# confusion matrix
fig, ax = plt.subplots(figsize=(7.5, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    cmap="Blues",
    xticks_rotation=45,
    ax=ax,
    colorbar=False,
)
ax.set_title("Confusion matrix")
plt.tight_layout()
plt.show()


## 7. Demo helpers

After training I can:

- predict a shape
- get confidence from `predict_proba`
- pull top TF-IDF words as keywords
- find a few similar old reports (cosine similarity)


In [ ]:
def extract_keywords(text, model, top_k=6):
    # top words for this report
    tfidf = model.named_steps["tfidf"]
    vec = tfidf.transform([text])
    if vec.nnz == 0:
        return []

    names = np.array(tfidf.get_feature_names_out())
    row = vec.tocoo()
    ranked = sorted(zip(row.col, row.data), key=lambda t: t[1], reverse=True)

    keywords = []
    for idx, score in ranked:
        term = names[idx]
        # skip bigrams so the list is cleaner
        if " " in term:
            continue
        keywords.append(term)
        if len(keywords) >= top_k:
            break
    return keywords


def format_location(city, state, country=""):
    city = str(city or "").strip().title()
    state = str(state or "").strip().upper()
    country = str(country or "").strip().lower()
    if city and state:
        return f"{city}, {state}"
    if city:
        return city
    return "Unknown location"


# sample of training reports for similar reports
rng = np.random.default_rng(42)
n_sim = min(10000, len(X_train))
sim_idx = rng.choice(len(X_train), size=n_sim, replace=False)
sim_texts = X_train.iloc[sim_idx].values
sim_meta = meta_train.iloc[sim_idx].reset_index(drop=True)
sim_matrix = model.named_steps["tfidf"].transform(sim_texts)


def find_similar(text, top_k=3):
    # closest reports by cosine sim
    q = model.named_steps["tfidf"].transform([text])
    sims = cosine_similarity(q, sim_matrix).ravel()
    order = np.argsort(-sims)

    out = []
    for i in order[:top_k]:
        if sims[i] <= 0:
            break
        row = sim_meta.iloc[int(i)]
        year = row["year"]
        if pd.notna(year):
            year = int(year)
        else:
            year = None
        out.append({
            "location": format_location(row["city"], row["state"], row["country"]),
            "year": year,
            "shape": row["shape"],
            "snippet": str(row["comments"])[:120],
            "similarity": float(sims[i]),
        })
    return out


def analyze_report(text):
    # main demo function
    text = clean_text(text)
    if len(text) < 5:
        return {"error": "report is too short"}

    probs = model.predict_proba([text])[0]
    classes = model.named_steps["clf"].classes_
    best = int(np.argmax(probs))

    return {
        "predicted_shape": str(classes[best]),
        "confidence": float(probs[best]),
        "keywords": extract_keywords(text, model),
        "similar_reports": find_similar(text),
        "class_probabilities": {
            str(c): float(p)
            for c, p in sorted(zip(classes, probs), key=lambda t: -t[1])
        },
    }


def print_card(result):
    if result.get("error"):
        print(result["error"])
        return

    print("Predicted Shape")
    print(result["predicted_shape"].title())
    print()
    print("Confidence")
    print(f"{result['confidence'] * 100:.0f}%")
    print()
    print("Most Similar Reports")
    for s in result["similar_reports"]:
        year = f" ({s['year']})" if s.get("year") else ""
        print(f"- {s['location']}{year}")
    print()
    print("Common Keywords")
    for k in result["keywords"]:
        print(k)


# quick tests
demos = [
    "three bright lights in a triangle, silent, hovering over the road",
    "orange ball of fire streaked across the sky and broke apart",
    "long metallic tube, no wings, moved steadily without sound",
    "round silver disc tilted on edge above the treeline",
]

for d in demos:
    print("=" * 50)
    print("INPUT:", d)
    print("-" * 50)
    print_card(analyze_report(d))
    print()


## 8. Try your own report

Change the text below and re-run this cell.


In [ ]:
my_report = """
I was driving west of the city when I noticed a large dark triangular object
with three white lights, one at each corner. It hovered silently for nearly a
minute, then accelerated away without any sound of engines.
"""

result = analyze_report(my_report)
print_card(result)

print()
print("All class probabilities:")
for shape, p in result.get("class_probabilities", {}).items():
    print(f"  {shape:12s} {p * 100:5.1f}%")


## Summary

Full supervised workflow on real text data:

load → clean → explore → split → TF-IDF + Naive Bayes → evaluate → demo

**Limits:** short comments, noisy labels, class imbalance (`light` is huge), and this is just a classic baseline model not some big LLM.

**Disclaimer:** this classifies dataset labels from text. It does not verify if any sighting is real.
